In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import pybiomart
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad
import warnings
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
from typing import Literal
from anndata import AnnData
from anndata import ImplicitModificationWarning        
from scipy.stats import pearsonr, spearmanr, kendalltau
from scipy.sparse import csr_matrix, issparse
from tqdm.auto import tqdm

from utils_correlations import get_gene_conversion_dict, replace_greek, inverse_replace_greek, compare_fcs, create_heatmap, get_mouse_dict, map_gene_names
from utils_name_maps import cytokine_name_map

In [ ]:
FIGDIR = "./figures_new"
os.makedirs(FIGDIR, exist_ok=True)
mouse_color = "#1B9E77"
human_color = "#7570B3"
cross_color = "#D95F02"

label_color_map = {
    "mouse": mouse_color,
    "human": human_color,
}

In [ ]:
celltype_combinations = [
    # (mouse, human)
    ('T_cell_CD4', "CD4"),
    ('T_cell_CD8', "CD8"),
    ('B_cell', "B"),
    ('Treg', 'Treg'),
    ("ILC", "ILC"),
    ("Monocyte", "CD14_Mono"),
    ("Monocyte", "CD16_Mono"),
    ("NK_cell", "CD56-dim_NK"),
    ("NK_cell", "CD56-bright_NK"),
    ("pDC", "pDC"),
    ("cDC1", "cDC"),
    ("cDC2", "cDC"),
]

title_map = {
    "T_cell_CD4__CD4": "CD4 T cell",
    "T_cell_CD8__CD8": "CD8 T cell",
    "B_cell__B": "B cell",
    "Treg__Treg": "Treg",
    "ILC__ILC": "ILC",
    "Monocyte__CD14_Mono": "CD14 Monocyte / Monocyte",
    "Monocyte__CD16_Mono": "CD16 Monocyte / Monocyte",
    "NK_cell__CD56-dim_NK": "NK CD56low / NK cell",
    "NK_cell__CD56-bright_NK": "NK CD56hi / NK cell",
    "pDC__pDC": "pDC",
    "cDC1__cDC": "cDC / cDC1",
    "cDC2__cDC": "cDC / cDC2",
}

title_map_short = {
    "T_cell_CD4__CD4": "CD4 T cell",
    "T_cell_CD8__CD8": "CD8 T cell",
    "B_cell__B": "B cell",
    "Treg__Treg": "Treg",
    "ILC__ILC": "ILC",
    "Monocyte__CD14_Mono": "CD14 Mono",
    "Monocyte__CD16_Mono": "CD16 Mono",
    "NK_cell__CD56-dim_NK": "NK CD56low",
    "NK_cell__CD56-bright_NK": "NK CD56hi",
    "pDC__pDC": "pDC",
    "cDC1__cDC": "cDC1",
    "cDC2__cDC": "cDC2",
}

celltype_color_mapping = {
    'B': "#2278B5",
    'B cell': "#2278B5",
    'B_cell__B': "#2278B5",
    
    'B Naive': "#2278B5",
    'Naive B cell': "#2278B5",
    
    'B Intermediate/Memory': "#42B9EB",
    'Intermediate B cell': "#42B9EB",

    'CD4': "#F57F20",
    'CD4 T cell': "#F57F20",
    'T_cell_CD4__CD4': "#F57F20",
    'CD4 Naive': "#F7901E",
    'CD4 Naive T cell': "#F7901E",
    'CD4 Memory': "#F15A22",
    'CD4 Memory T cell': "#F15A22",

    'CD8': "#2FA148",
    'CD8 T cell': "#2FA148",
    'T_cell_CD8__CD8': "#2FA148",
    'CD8 Naive': "#2FA148", #"#73BF98", #"#74AC42",
    'CD8 Naive T cell': "#2FA148", #"#73BF98", #"#74AC42",
    'CD8 Memory': "#17A57F",
    'CD8 Memory T cell': "#17A57F",
    
    'Treg': "#7750A1",
    'Treg__Treg': "#7750A1",
    'MAIT': "#4C87C6",
    'NKT': "#ED2224",
    'NK CD56hi': "#F9BFCB",
    'CD56-bright_NK': "#F9BFCB",
    'CD56-bright NK': "#F9BFCB",
    'NK_cell__CD56-bright_NK': "#8C471F",
    'NK CD56low': "#8C471F",
    'CD56-dim_NK': "#8C471F",
    'NK_cell__CD56-dim_NK': "#8C471F",
    'CD56-dim NK': "#8C471F",
    'ILC': "#282A74",
    'ILC__ILC': "#282A74",
    'pDC': "#ED2A91",
    'pDC__pDC': "#ED2A91",
    'cDC': "#6ABD45",
    'cDC1__cDC': "#6ABD45",
    'cDC1': "#6ABD45",
    'cDC2__cDC': "#6ABD45",
    'cDC2': "#6ABD45",
    'CD14 Mono': "#7D277D",
    'CD14_Mono': "#7D277D",
    'Monocyte__CD14_Mono': "#7D277D",
    'CD16 Mono': "#E4C494",
    'CD16_Mono': "#E4C494",
    'Monocyte__CD16_Mono': "#E4C494",
    'HSPC': "#6A6A6A",
    "Granulocyte": "#FDD700",
    "other": "#FDD700",
    "Plasmablast": "#556B30",
    # "Platelet": "#FFFFFF",
    "Platelet": '#d9d9d9',
    
}

In [ ]:
def count_unique_non_rep_donors(df, species: Literal["mouse", "human"]):
    df_long = df.melt(id_vars=['cytokine'], value_vars=['donor1', 'donor2'], value_name='num_donors')
    df_long = df_long.dropna(subset=['num_donors'])
    if species == "human":
        df_filtered = df_long[~df_long['num_donors'].str.lower().str.startswith('rep')].copy()
    elif species == "mouse":
        df_filtered = df_long[~df_long['num_donors'].str.lower().str.startswith('donor')].copy()
    else:
        raise ValueError()
    unique_donor_counts = df_filtered.groupby('cytokine')['num_donors'].nunique().sort_values(ascending=False).to_frame()
    
    return unique_donor_counts


def add_missing_values(df, idcs, value, index_name, value_name):
    missing_idcs = np.array(idcs)[~np.isin(idcs, df.index)]
    if len(missing_idcs) == 0:
        return df
    _df = pd.DataFrame()\
        .from_dict(
            zip(missing_idcs, np.ones_like(missing_idcs)*value)
        )\
        .rename({0: index_name, 1: value_name}, axis=1)\
        .set_index(index_name)
    return pd.concat([df, _df], axis=0)

In [ ]:
human_to_mouse_dict, mouse_to_human_dict = get_gene_conversion_dict()

In [ ]:
adata_mouse = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all.h5ad")
cytokines_mouse = adata_mouse.obs.cytokine.apply(inverse_replace_greek).unique().tolist() 

In [ ]:
hcd = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv", index_col=0)
cytokines_human = hcd.cytokine.apply(inverse_replace_greek).unique()

In [ ]:
shared_cytokines = np.intersect1d(cytokines_mouse, cytokines_human)
print(len(shared_cytokines))

In [ ]:
cytokines_human[~np.isin(cytokines_human, cytokines_mouse)]

In [ ]:
cytokines_mouse = np.array(cytokines_mouse)
cytokines_mouse[~np.isin(cytokines_mouse, cytokines_human)]

In [ ]:
shared_cytokines

In [ ]:
hcd.celltype.unique()

In [ ]:
celltype_combinations

In [ ]:
hcd_sig = hcd.loc[
    (
        (hcd.adj_p_value < 0.05)
        &(hcd.log_fc.abs() > 0.25)
        &(hcd.num_DE_pbs_wells > 3)
    )
]

In [ ]:
# hcd.loc[:, "celltype_old"] = 
name_map = {
    'CD4_T_cell': "CD4",
    'CD8_T_cell': 'CD8',
    'B_cell': 'B',
    'NK_CD56low': 'CD56-dim_NK',
    'NK_CD56hi': 'CD56-bright_NK',
}
hcd_sig.loc[:, "celltype_old"] = hcd_sig.loc[:, "celltype"].apply(lambda x: name_map.get(x,x))

In [ ]:
def get_DE_genes(hcd_sig, species, celltype_combination, cytokine, mouse_to_human_dict=None, cytokine_name_map=None):
    if species is None:
        return None
    if species in ["human", "both"]:
        geneset_restriction_human = hcd_sig.loc[((hcd_sig.celltype_old == celltype_combination[1]) & (hcd_sig.cytokine == cytokine))].gene.values
        if species == "human":
            return ("human", geneset_restriction_human)
    if species in ["mouse", "both"]:

        mcd = get_mouse_dict(
            cytokine_name_map=cytokine_name_map,
            celltype=celltype_combination[0],
            cytokine=cytokine,
            mouse_to_human_dict=mouse_to_human_dict, 
            subset_to_homologs=True,
        )
        
        geneset_restriction_mouse = mcd.loc[((mcd.celltype == celltype_combination[0]) & (mcd.cytokine == cytokine))].gene.values
        if species == "mouse":
            return ("mouse", geneset_restriction_mouse)
    return ("both", np.intersect1d(geneset_restriction_human, geneset_restriction_mouse))

In [ ]:
all_corrs = []

for celltype_combination in tqdm(celltype_combinations[:]):
    for cytokine in tqdm(shared_cytokines, leave=False):
        for geneset_restriction in [None, "both", "human", "mouse"]:

            geneset_restriction = get_DE_genes(
                hcd_sig=hcd_sig, 
                species=geneset_restriction, 
                celltype_combination=celltype_combination, 
                cytokine=cytokine,
                mouse_to_human_dict=mouse_to_human_dict,
                cytokine_name_map=cytokine_name_map,
            )

            if geneset_restriction is None:
                thresholds_expression = [0.1, 0.2, 0.5, 1]
                methods_subset_expressed_genes_by = ["union", "intersection"]
                
            else:
                thresholds_expression = [-1] # use only DE as gene filter
                methods_subset_expressed_genes_by = ["union"]
                # does not really matter as expression_threshold is -1
            
            for threshold_expression in thresholds_expression:
                for mean_per_species in [False, True]:
                    for method in ["pearson", "spearman"]:
                        for subset_expressed_genes_by in methods_subset_expressed_genes_by:
                            output = compare_fcs(
                                celltype_combination=celltype_combination, 
                                cytokine=cytokine, 
                                threshold_expression=threshold_expression,
                                mean_per_species=mean_per_species,
                                method=method,
                                mouse_to_human_dict=mouse_to_human_dict,
                                threshold_min_num_cells=20,
                                apply_log2 = True,
                                geneset_restriction = geneset_restriction,
                                subset_expressed_genes_by=subset_expressed_genes_by,
                            )
                            if output is not None:
                                corrs = output["corrs"]
                                all_corrs.append(corrs)
                        
                    
                
            
                    
# all_corrs = pd.concat(all_corrs, axis=0)
# filename = "./human_mouse_correlations_final_all_genes.csv"
# all_corrs.to_csv(filename)

In [ ]:
all_corrs = pd.read_csv("./human_mouse_correlations_final_all_genes.csv", index_col=0)
all_corrs

### create three subsets: human_vs_mouse, human_vs_human, mouse_vs_mouse

In [ ]:
all_corrs_hm = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.apply_log2==True)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (
            (all_corrs.donor1.str.startswith("D") & all_corrs.donor2.str.startswith("r")) 
            | (all_corrs.donor2.str.startswith("D") & all_corrs.donor1.str.startswith("r"))
        )
    )
]
all_corrs_hm.loc[:, "cytokine"] = all_corrs_hm.loc[:, "cytokine"].apply(replace_greek)


all_corrs_mm = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == False)
        & (all_corrs.apply_log2==True)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.donor1.str.startswith("r") & all_corrs.donor2.str.startswith("r"))
    )
]
all_corrs_mm = all_corrs_mm.loc[
    (
        (all_corrs_mm.donor1 != all_corrs_mm.donor2)
        &(all_corrs_mm.donor1.str.replace("rep", "").astype(int) < all_corrs_mm.donor2.str.replace("rep", "").astype(int))
    )
]
all_corrs_mm.loc[:, "cytokine"] = all_corrs_mm.loc[:, "cytokine"].apply(replace_greek)


all_corrs_hh = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == False)
        & (all_corrs.apply_log2==True)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.donor1.str.startswith("D") & all_corrs.donor2.str.startswith("D"))
    )
]
all_corrs_hh = all_corrs_hh.loc[
    (
        (all_corrs_hh.donor1 != all_corrs_hh.donor2)
        &(all_corrs_hh.donor1.str.replace("Donor", "").astype(int) < all_corrs_hh.donor2.str.replace("Donor", "").astype(int))
    )
]
all_corrs_hh.loc[:, "cytokine"] = all_corrs_hh.loc[:, "cytokine"].apply(replace_greek)

In [ ]:
handles = []
labels = []

for label, color in label_color_map.items():
    patch = mpatches.Patch(color=color, label=label)
    handles.append(patch)
    labels.append(label)

fig, ax = plt.subplots(figsize=(0.5, 0.3))
legend = ax.legend(handles, labels, loc='center', frameon=False, fontsize=6) 

for text in legend.get_texts():
    text.set_fontfamily('Arial')

ax.axis('off')
# plt.savefig(os.path.join(FIGDIR, f"legend_human_mouse.svg"), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
threshold_expression = -1

In [ ]:
_data = all_corrs_hm.loc[
    (
        (all_corrs_hm.threshold_expression==threshold_expression)
        &(all_corrs_hm.geneset_restriction == "both")
    )
]
order = _data.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.boxplot(
    data = _data,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
# plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
_data = all_corrs_hm.loc[all_corrs_hm.threshold_expression==threshold_expression]
order = _data.groupby("cytokine").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.105*len(order), 1.3))
sns.boxplot(
    data = _data,
    x="cytokine",
    y="correlation",
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
    color="#89CFF0",
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [replace_greek(s) for s in order], rotation=90, fontsize=6, family="Arial")
ax.grid(axis="y")
ax.set_xlabel("")
ax.set_ylabel("correlation", fontsize=6, family="Arial", labelpad=0)
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)

plt.title("Median correlations per cytokine", fontsize=6, family="Arial", y=1, pad=-5)
#plt.savefig(os.path.join(FIGDIR, f"boxplot_per_cytokine_{threshold_expression}.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
add_barplots = False

combos = [
    ("T_cell_CD4__CD4"),
    ("Monocyte__CD16_Mono"),
]

vmin = all_corrs_hm.loc[all_corrs_hm.celltype_combination.isin(combos) & (all_corrs_hm.threshold_expression==threshold_expression)].num_genes.min()
vmax = all_corrs_hm.loc[all_corrs_hm.celltype_combination.isin(combos) & (all_corrs_hm.threshold_expression==threshold_expression)].num_genes.max()

# for celltype_combo in sorted(all_corrs_hm.celltype_combination.unique()):
for celltype_combo in sorted(combos):

    _data = all_corrs_hm.loc[
        (
            (all_corrs_hm.celltype_combination==celltype_combo)
            & (all_corrs_hm.threshold_expression==threshold_expression)
        )
    ]
    corrs_hh = all_corrs_hh.loc[
        (
            (all_corrs_hh.celltype_combination==celltype_combo)
            & (all_corrs_hh.threshold_expression==threshold_expression)
        )
    ]
    corrs_mm = all_corrs_mm.loc[
        (
            (all_corrs_mm.celltype_combination==celltype_combo)
            & (all_corrs_mm.threshold_expression==threshold_expression)
        )
    ]

    
    group_means = _data.num_genes.unique()
    cmap = sns.color_palette("light:#F4D03F", as_cmap=True)
    
    # norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    # vmin = _data.num_genes.min()
    # vmax = _data.num_genes.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colors = [cmap(norm(v)) for v in group_means]
    order = _data.groupby("cytokine").correlation.median().sort_values(ascending=False).index

    num_humans = count_unique_non_rep_donors(corrs_hh, "human")
    num_humans = add_missing_values(num_humans, order, 1, "cytokine", "num_donors")
    num_humans.loc[:, "species"] = "human"
    num_mice = count_unique_non_rep_donors(corrs_mm, "mouse")
    num_mice = add_missing_values(num_mice, order, 1, "cytokine", "num_donors")
    num_mice.loc[:, "species"] = "mouse"
    num_samples = pd.concat([num_humans.reset_index(), num_mice.reset_index()], axis=0, ignore_index=True)

    if add_barplots:
        fig, ax = plt.subplots(2,1,figsize=(0.105*len(order), 2), height_ratios=[0.2, 0.8], sharex=True)
        ax_top = ax[0]
        ax_bottom = ax[1]
    else:
        fig, ax = plt.subplots(1,1,figsize=(0.105*len(order), 1.3))
        ax_bottom = ax
        
    sns.boxplot(
        data = _data,
        x="cytokine",
        y="correlation",
        hue="num_genes",
        order = order,
        ax=ax_bottom,
        palette=colors,
        # palette="dark:#E7298A",
        # palette=cmap,
        linewidth=0.5,
        linecolor="black",
        legend=None,
        flierprops={
            'marker': 'o',
            'markerfacecolor': 'gray',
            'markeredgecolor': 'black',
            'markersize': 1,
            'linewidth': 0.5,
        }
        
    )
    sns.pointplot(
        corrs_hh,
        order=order,
        x="cytokine",
        y="correlation",
        markers=".",
        markersize=3,
        linewidth=0.5,
        color=human_color,
        estimator="median",
        errorbar=("pi", 50),
        ax=ax_bottom,
    )
    sns.pointplot(
        corrs_mm,
        x="cytokine",
        y="correlation",
        markers=".",
        markersize=3,
        linewidth=0.5,
        color=mouse_color,
        estimator="median",
        errorbar=("pi", 50),
        ax=ax_bottom,
    )
    if add_barplots:
        sns.barplot(
            data=num_samples,
            x="cytokine",
            y="num_donors",
            hue="species",
            order=order,
            ax=ax_top,
            palette={"human": human_color, "mouse": mouse_color},
            legend=False,
        )
    
        ax_top.set_ylim([0, 13])
        ax_top.set_xlabel("")
        ax_top.set_ylabel("# samples", fontsize=6, family="Arial")
        ax_top.set_yticks([0, 3, 6, 9, 12], ["0", "", "6", "", "12"], fontsize=6, family="Arial")
        ax_top.grid(axis="y")
        ax_top.spines[['right', 'top']].set_visible(False)
        ax_top.set_axisbelow(True)
        ax_top.set_title(title_map[celltype_combo], fontsize=6, family="Arial")#, y=1, pad=0)
    else:
        ax_bottom.set_title(title_map[celltype_combo], fontsize=6, family="Arial")#, y=1, pad=0)
    
    # plt.ylim([])
    y_margin = 0.02
    ax_bottom.margins(y=y_margin)
    ax_bottom.set_xticks(np.arange(len(order)), [str(s) for s in order], rotation=90, fontsize=6, family="Arial")
    yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
    ax_bottom.set_yticks(yticks, [str(_) for _ in yticks], fontsize=6, family="Arial")
    ax_bottom.grid(axis="y")
    ygridlines = ax.get_ygridlines()
    # Find the line at the top (highest y value)
    top_y = max([tick.get_ydata()[0] for tick in ygridlines])
    for line in ygridlines:
        # Hide the line at the top
        if line.get_ydata()[0] == top_y:
            line.set_visible(False)
    ax_bottom.set_xlabel("")
    ax_bottom.set_ylabel("correlation", fontsize=6, family="Arial", labelpad=0)
    ax_bottom.spines[['right', 'top']].set_visible(False)
    ax_bottom.margins(y=y_margin)
    ax_bottom.set_ylim([-0.32, 1])

    if False:
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        cbar = plt.colorbar(sm, ax=ax_bottom, pad=0.01)
        cbar.set_label("num shared genes", rotation=270, labelpad=15, family="Arial", fontsize=6)  # customize label
        cbar.ax.tick_params(labelsize=6)
        for label in cbar.ax.get_yticklabels():
            label.set_fontfamily('Arial')
        # plt.tight_layout()

    filename = f"boxplot_{threshold_expression}_{celltype_combo}.svg"
    if add_barplots:
        filename = filename.replace(".svg", "_w_bars.svg")
    plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
    plt.show()

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array(np.array([]))
    
    fig, ax = plt.subplots(1, 1, figsize=(0.1, 1))
    cbar = plt.colorbar(sm, cax=ax)
    
    cbar.set_label("# Genes", rotation=270, labelpad=15, family='Arial', fontsize=6)
    cbar.ax.tick_params(labelsize=6)
    cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
    
    for label in cbar.ax.get_yticklabels():
        label.set_fontfamily('Arial')
    
    plt.tight_layout()
    filename = f"colorbar_num_genes_{threshold_expression}_{celltype_combo}.svg"
    if add_barplots:
        filename = filename.replace(".svg", "_w_bars.svg")
    #plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
    plt.show()

    # break

In [ ]:
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(np.array([]))

fig, ax = plt.subplots(1, 1, figsize=(1, 0.1))
cbar = plt.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("# Genes", rotation=0, labelpad=-3, family='Arial', fontsize=6)
cbar.ax.xaxis.set_ticks_position('top')
cbar.ax.xaxis.set_label_position('top')
cbar.ax.set_xticks([200, 360], ["200",  "360"], family="Arial")
cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)

for label in cbar.ax.get_yticklabels():
    label.set_fontfamily('Arial')

plt.tight_layout()
filename = f"colorbar_num_genes_{threshold_expression}.svg"
plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm = mcolors.Normalize(vmin=-1, vmax=1))
sm.set_array(np.array([]))

fig, ax = plt.subplots(1, 1, figsize=(1, 0.1))
cbar = plt.colorbar(sm, cax=ax, orientation="horizontal")

cbar.set_label("Pearson corr.", rotation=0, labelpad=-5, family='Arial', fontsize=6)
cbar.ax.set_xticks([-1, 1], ["-1", "1"], family="Arial")
cbar.ax.tick_params(labelsize=6)
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)

for label in cbar.ax.get_yticklabels():
    label.set_fontfamily('Arial')

plt.tight_layout()
filename = f"colorbar_correlations.svg"
plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "IL-7"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "CD30L"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "IFN-epsilon"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "Monocyte__CD14_Mono"
cytokine = "IFN-beta"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]

corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

#display(corrs_human_and_mouse)

In [ ]:
all_corrs

In [ ]:
tmp = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
        & (all_corrs.donor1.str.startswith("D"))
        & (all_corrs.donor2.str.startswith("D"))
    )
]
tmp = tmp.loc[tmp.apply(lambda x: int(x.donor1.replace("Donor", "")) < int(x.donor2.replace("Donor", "")), axis=1)].sort_values(["donor1", "donor2"])

In [ ]:
tmp.correlation.min(), tmp.correlation.max(), tmp.correlation.mean(), tmp.correlation.median()

In [ ]:
tmp2 = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
        & (all_corrs.donor1.str.startswith("D"))
        & (all_corrs.donor2.str.startswith("r"))
    )
]

In [ ]:
tmp2.correlation.min(), tmp2.correlation.max(), tmp2.correlation.mean(), tmp2.correlation.median()

In [ ]:
tmp3 = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
        & (all_corrs.donor1.str.startswith("r"))
        & (all_corrs.donor2.str.startswith("r"))
    )
]
tmp3 = tmp3.loc[tmp3.apply(lambda x: int(x.donor1.replace("rep", "")) < int(x.donor2.replace("rep", "")), axis=1)].sort_values(["donor1", "donor2"])

In [ ]:
tmp3.correlation.min(), tmp3.correlation.max(), tmp3.correlation.mean(), tmp3.correlation.median()

In [ ]:
celltype = "cDC1__cDC"
cytokine = "IFN-beta"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "NK_cell__CD56-dim_NK"
cytokine = "IL-15"
threshold_expression = 0.5

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "NK_cell__CD56-bright_NK"
cytokine = "IL-15"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "Monocyte__CD16_Mono"
cytokine = "IFN-beta"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "IFN-beta"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "G-CSF"
threshold_expression = 0.5

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = "T_cell_CD4__CD4"
cytokine = "GM-CSF"
threshold_expression = 0.2

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
]
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
if False:
    [('T_cell_CD4', 'CD4'),
     ('T_cell_CD8', 'CD8'),
     ('B_cell', 'B'),
     ('Treg', 'Treg'),
     ('ILC', 'ILC'),
     ('Monocyte', 'CD14_Mono'),
     ('Monocyte', 'CD16_Mono'),
     ('NK_cell', 'CD56-dim_NK'),
     ('NK_cell', 'CD56-bright_NK'),
     ('pDC', 'pDC'),
     ('cDC1', 'cDC'),
     ('cDC2', 'cDC')]

celltype = 'B_cell__B'
cytokine = "IL-10"
threshold_expression = -1

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.geneset_restriction=="both")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
].drop_duplicates()
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = 'cDC2__cDC'
cytokine = "IL-13"
threshold_expression = -1

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.geneset_restriction=="both")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
].drop_duplicates()
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

In [ ]:
celltype = 'cDC1__cDC'
cytokine = "IL-13"
threshold_expression = -1

corrs_human_and_mouse = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.cytokine == cytokine) 
        & (all_corrs.celltype_combination==celltype)
        & (all_corrs.mean_per_species == False) 
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.geneset_restriction=="both")
        & (all_corrs.threshold_expression==threshold_expression)
        & (all_corrs.apply_log2==True)
    )
].drop_duplicates()
corrs_human_and_mouse = corrs_human_and_mouse.pivot(
    index="donor1", columns="donor2", values="correlation"
)
_title = f"{replace_greek(cytokine)} in {title_map[celltype]}"
ax = create_heatmap(corrs_human_and_mouse, fontsize=6, figsize=(1.3, 1.3), annot=False, path = os.path.join(FIGDIR, f"heatmap_{celltype}_{cytokine}_{threshold_expression}.svg"), title=_title, label_color_map=label_color_map)

### subset to strong responses

In [ ]:
def get_strong_responses_human() -> pd.DataFrame:
    celltype_map = {
        'CD4': "CD4_T_cell",
        'CD8': "CD4_T_cell",
        'B': "B_cell" ,
        'Treg': "Treg",
        'CD14_Mono': "CD14_Mono",
        'CD16_Mono': "CD16_Mono",
        'CD56-dim_NK': "NK_CD56low",
        'CD56-bright_NK': "NK_CD56hi",
        'pDC': "pDC",
        'cDC': "cDC",
    }
    strong_magnitude_response_human = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/SI_tables.xlsx", sheet_name="3.strong_impact")
    dict_strong_responses_human = {}
    for celltype in strong_magnitude_response_human.drop("Cytokine", axis=1).columns:
        dict_strong_responses_human[celltype.replace(" ", "_")] = strong_magnitude_response_human.loc[strong_magnitude_response_human.loc[:, celltype], "Cytokine"].values
    df_strong_responses_human = pd.DataFrame.from_dict(
        data={
            key: len(dict_strong_responses_human[key]) 
            for key in dict_strong_responses_human.keys()
        },
        columns=["num_cytokines"],
        orient="index"
    )
    df_strong_responses_human = df_strong_responses_human.reset_index().rename({"index": "celltype"}, axis=1)
    df_strong_responses_human.loc[:, "cytokines"] = df_strong_responses_human.celltype.apply(lambda x: dict_strong_responses_human[x])
    df_strong_responses_human.loc[:, "celltype"] = df_strong_responses_human.celltype.str.replace("_", " ")
    df_strong_responses_human.loc[:, "species"] = "human"
    df_strong_responses_human = df_strong_responses_human.loc[~df_strong_responses_human.celltype.isin(["MAIT", "NKT", "HSPC"])]
    df_strong_responses_human.loc[:, "celltype_coarse"] = df_strong_responses_human.celltype.map(
        {
            "B cell": "B cell",
            "CD4 T cell": "CD4 T cell",
            "CD8 T cell": "CD8 T cell",
            "Treg": "Treg",
            "CD14 Mono": "Monocyte",
            "CD16 Mono": "Monocyte",
            "NK CD56hi": "NK cell",
            "NK CD56low": "NK cell",
            "Treg": "Treg",
            "cDC": "cDC",
        }
    )
    df_strong_responses_human.rename({column: column + "_human" for column in df_strong_responses_human if not column in ["species", "celltype_coarse"]}, axis=1, inplace=True)
    return df_strong_responses_human

def load_response_magnitudes_mouse():
    df = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/41586_2023_6816_MOESM13_ESM.xlsx", sheet_name="Fig1d")
    df.loc[:, "Magnitude_norm_float"] = df.loc[:, "Magnitude_norm"]
    df.loc[df.Magnitude_norm_float == "≥100", "Magnitude_norm_float"] = 100
    df["Magnitude_norm_float"] = df.Magnitude_norm_float.astype(float)
    t = dict(df.groupby("Celltype").Magnitude_norm_float.quantile(0.35))
    df.loc[:, "in35perc"] = df.apply(lambda row: row.Magnitude_norm_float < t[row["Celltype"]], axis=1)
    thresholds = dict(df.loc[df.in35perc].groupby("Celltype").Magnitude_norm_float.mean() * 3)
    df.loc[:, "strong_impact"] = df.apply(lambda row: row["Magnitude_norm_float"] > thresholds[row["Celltype"]], axis=1)
    return df

def get_strong_responses_mouse() -> pd.DataFrame:
    df = load_response_magnitudes_mouse()
    dict_strong_responses_mouse = dict(df.loc[df.strong_impact].groupby(["Celltype"]).Cytokine.unique())
    df_strong_responses_mouse = pd.DataFrame.from_dict(
        data={
            key: len(dict_strong_responses_mouse[key])
            for key in dict_strong_responses_mouse.keys()
        },
        columns=["num_cytokines"],
        orient="index"
    )
    df_strong_responses_mouse = df_strong_responses_mouse.reset_index().rename({"index": "celltype"}, axis=1)
    df_strong_responses_mouse.loc[:, "cytokines"] = df_strong_responses_mouse.celltype.apply(lambda x: dict_strong_responses_mouse[x])
    name_map = {
        "B cell": "B cell",
        "CD4+ T cell": "CD4 T cell",
        "CD8+ T cell": "CD8 T cell",
        "Monocyte": "Monocyte",
        "NK cell": "NK cell",
        "Treg": "Treg",
        "cDC1": "cDC1",
        "cDC2": "cDC2",
    }
    df_strong_responses_mouse.loc[:, "celltype"] = df_strong_responses_mouse.celltype.apply(lambda x: name_map.get(x,None))
    df_strong_responses_mouse = df_strong_responses_mouse.loc[~df_strong_responses_mouse.celltype.isna()]
    df_strong_responses_mouse.loc[:, "species"] = "mouse"
    df_strong_responses_mouse.loc[:, "celltype_coarse"] = df_strong_responses_mouse.celltype.map(
        {
            "B cell": "B cell",
            "CD4 T cell": "CD4 T cell",
            "CD8 T cell": "CD8 T cell",
            "Treg": "Treg",
            "Monocyte": "Monocyte",
            "NK cell": "NK cell",
            "Treg": "Treg",
            "cDC1": "cDC",
            "cDC2": "cDC",
        }
    )
    df_strong_responses_mouse.rename({column: column + "_mouse" for column in df_strong_responses_mouse if not column in ["species", "celltype_coarse"]}, axis=1, inplace=True)
    return df_strong_responses_mouse


def plot_response_distributions(df_strong_responses_mouse):
    for celltype in df_strong_responses_mouse.Celltype.unique():
        fig, ax = plt.subplots(1,1,figsize=(2,2))
        values = df_strong_responses_mouse.loc[df_strong_responses_mouse.Celltype == celltype].Magnitude_norm_float
        ax.hist(values, bins=20)
        t = df_strong_responses_mouse.loc[
            (
                (df_strong_responses_mouse.Celltype == celltype)
                &(df_strong_responses_mouse.strong_impact)
            )
        ].Magnitude_norm_float.min()
        
        plt.vlines(t, ymin=0, ymax=plt.ylim()[1], color="r", linestyle="--")
        plt.title(f"{celltype} (n={len(values)})")
        plt.xlabel("response magnitude")
        plt.ylabel("# cytokines")
        plt.show()
        # break

In [ ]:
df_response_magnitudes_mouse = load_response_magnitudes_mouse()
plot_response_distributions(df_response_magnitudes_mouse)

In [ ]:
df_strong_responses_mouse = get_strong_responses_mouse()

In [ ]:
df_strong_responses_human = get_strong_responses_human()

merged_df = pd.merge(
    df_strong_responses_human,
    df_strong_responses_mouse,
    on='celltype_coarse',
    how='outer'
)
merged_df.loc[:, "overlap"] = merged_df.apply(lambda x: len(np.intersect1d(x.cytokines_human, x.cytokines_mouse)), axis=1)
merged_df.loc[:, "overlap_cytokines"] = merged_df.apply(lambda x: np.intersect1d(x.cytokines_human, x.cytokines_mouse), axis=1)
merged_df.loc[:, "celltype_combo"] = merged_df.celltype_human + "__" + merged_df.celltype_mouse
a = merged_df.drop([c for c in merged_df.columns if "mouse" in c], axis=1).rename({c: c.replace("_human", "") for c in merged_df.columns}, axis=1)
b = merged_df.drop([c for c in merged_df.columns if "human" in c], axis=1).rename({c: c.replace("_mouse", "") for c in merged_df.columns}, axis=1)
b.loc[:, "species_x"] = b.loc[:, "species_y"]
df_strong_responses = pd.concat([a, b], axis=0).drop("species_y", axis=1).rename({"species_x": "species"}, axis=1)
_overlap = df_strong_responses.loc[df_strong_responses.species == "human"]
_overlap.loc[:, "species"] = "overlap"
_overlap.loc[:, "num_cytokines"] = _overlap.loc[:, "overlap"]
df_strong_responses = pd.concat([df_strong_responses, _overlap], axis=0)
df_strong_responses

In [ ]:
print(pearsonr(merged_df.num_cytokines_human, merged_df.num_cytokines_mouse)[0])
print(spearmanr(merged_df.num_cytokines_human, merged_df.num_cytokines_mouse)[0])
print(merged_df.overlap.min(), merged_df.overlap.max())

In [ ]:
label_color_map = {
    "human": human_color,
    "mouse": mouse_color,
    "overlap": "gold",
}
order_strong_responses = df_strong_responses.loc[df_strong_responses.species.isin(["human", "mouse"])].groupby("celltype_combo")["num_cytokines"].max().sort_values(ascending=False).index

fig, ax = plt.subplots(1,1,figsize=(2.5, 1.3))
sns.barplot(
    data=df_strong_responses,
    x="celltype_combo",
    y="num_cytokines",
    hue="species",
    legend=False,
    palette=label_color_map,
    order=order_strong_responses,
    gap=0,
)
plt.ylabel("# cytokines w. strong response", fontsize=6, family="Arial")
plt.xlabel("")

def clean_xlabel(l):
    if not isinstance(l, str):
        l = l.get_text()
    ls = l.split("__")
    if ls[0] == ls[1]:
        return ls[0]

    return f"{ls[1]} /\n{ls[0]}"
        
    # if len(ls[0]) > len(ls[1]):
    #    return f"{ls[1]} /\n{ls[0]}"
    #return f"{ls[0]} /\n{ls[1]}"

xticklabels = [clean_xlabel(l) for l in ax.get_xticklabels()]
ax.set_xticks(ax.get_xticks(), xticklabels, rotation=90, fontsize=6, family="Arial")
# ax.set_yticks(ax.get_yticks(), ax.get_yticklabels(), fontsize=6, family="Arial")
yticks = [0, 10, 20, 30, 40]
ax.set_yticks(yticks, [y for y in yticks], fontsize=6, family="Arial")
ax.grid(axis="y")
ax.spines[['right', 'top']].set_visible(False)
ax.set_ylim([0, 47])
plt.savefig(os.path.join(FIGDIR, f"barplot_strong_responses.svg"), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
df_strong_responses

In [ ]:
handles = []
labels = []

for label, color in label_color_map.items():
    patch = mpatches.Patch(color=color, label=label)
    handles.append(patch)
    labels.append(label)

fig, ax = plt.subplots(figsize=(0.5, 0.3))
legend = ax.legend(handles, labels, loc='center', frameon=False, fontsize=6, ncol=3) 

for text in legend.get_texts():
    text.set_fontfamily('Arial')

ax.axis('off')
plt.savefig(os.path.join(FIGDIR, f"barplot_strong_responses_legend.svg"), bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

In [ ]:
_clean_labels_mouse = {"T_cell_CD4": "CD4_T_cell", "T_cell_CD8": "CD8_T_cell"}
_clean_labels_human = {"CD4": "CD4_T_cell", "CD8": "CD8_T_cell", "CD56-bright_NK": "NK_CD56hi", "CD56-dim_NK": "NK_CD56low", "B": "B_cell"}
get_clean_label_mouse = lambda x: _clean_labels_mouse.get(x.split("__")[0], x.split("__")[0]).replace("_", " ")
get_clean_label_human = lambda x: _clean_labels_human.get(x.split("__")[1], x.split("__")[1]).replace("_", " ")

all_corrs_mm.celltype_combination.unique()

In [ ]:
all_corrs_mm_strong = all_corrs_mm.copy()
all_corrs_mm_strong = all_corrs_mm_strong.loc[all_corrs_mm_strong.celltype_combination.apply(get_clean_label_mouse).isin(df_strong_responses_mouse.celltype_mouse.values)]

all_corrs_mm_strong.loc[:, "strong_effect_mouse"] = all_corrs_mm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_mouse.set_index("celltype_mouse").loc[get_clean_label_mouse(row.celltype_combination)].cytokines_mouse, 
    axis=1
)

all_corrs_mm_strong.loc[:, "strong_effect_human"] = all_corrs_mm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_human.set_index("celltype_human").loc[get_clean_label_human(row.celltype_combination)].cytokines_human, 
    axis=1
)

In [ ]:
all_corrs_hm_strong = all_corrs_hm.copy()
all_corrs_hm_strong = all_corrs_hm_strong.loc[all_corrs_hm_strong.celltype_combination.apply(get_clean_label_mouse).isin(df_strong_responses_mouse.celltype_mouse.values)]
all_corrs_hm_strong.loc[:, "strong_effect_mouse"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_mouse.set_index("celltype_mouse").loc[get_clean_label_mouse(row.celltype_combination)].cytokines_mouse, 
    axis=1
)

all_corrs_hm_strong.loc[:, "strong_effect_human"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_human.set_index("celltype_human").loc[get_clean_label_human(row.celltype_combination)].cytokines_human, 
    axis=1
)

In [ ]:
all_corrs_hh_strong = all_corrs_hh.copy()
all_corrs_hh_strong = all_corrs_hh_strong.loc[all_corrs_hh_strong.celltype_combination.apply(get_clean_label_mouse).isin(df_strong_responses_mouse.celltype_mouse.values)]
all_corrs_hh_strong.loc[:, "strong_effect_mouse"] = all_corrs_hh_strong.apply(
    lambda row: row.cytokine in df_strong_responses_mouse.set_index("celltype_mouse").loc[get_clean_label_mouse(row.celltype_combination)].cytokines_mouse, 
    axis=1
)

all_corrs_hh_strong.loc[:, "strong_effect_human"] = all_corrs_hh_strong.apply(
    lambda row: row.cytokine in df_strong_responses_human.set_index("celltype_human").loc[get_clean_label_human(row.celltype_combination)].cytokines_human, 
    axis=1
)

In [ ]:
threshold_expression = -1

_data = all_corrs_hm_strong.loc[
    (
        (all_corrs_hm_strong.threshold_expression==threshold_expression)
        & (all_corrs_hm_strong.strong_effect_mouse == True)
        & (all_corrs_hm_strong.geneset_restriction == "both")
    )
]
order = _data.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.boxplot(
    data = _data,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
# ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
# plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
# plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}_strong_in_mouse.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
threshold_expression = -1

_data = all_corrs_hm_strong.loc[
    (
        (all_corrs_hm_strong.threshold_expression==threshold_expression)
        & (all_corrs_hm_strong.strong_effect_human == True)
        & (all_corrs_hm_strong.geneset_restriction == "both")
    )
]
order = _data.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.boxplot(
    data = _data,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
# plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
# plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}_strong_in_human.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
threshold_expression = -1

_data_hm = all_corrs_hm_strong.loc[
    (
        (all_corrs_hm_strong.threshold_expression==threshold_expression)
        & (all_corrs_hm_strong.strong_effect_human == True)
        & (all_corrs_hm_strong.strong_effect_mouse == True)
        & (all_corrs_hm_strong.geneset_restriction == "both")
    )
]

_data_hh = all_corrs_hh_strong.loc[
    (
        (all_corrs_hh_strong.threshold_expression==threshold_expression)
        & (all_corrs_hh_strong.strong_effect_human == True)
        & (all_corrs_hh_strong.strong_effect_mouse == True)
        & (all_corrs_hh_strong.geneset_restriction == "both")
    )
]

_data_mm = all_corrs_mm_strong.loc[
    (
        (all_corrs_mm_strong.threshold_expression==threshold_expression)
        & (all_corrs_mm_strong.strong_effect_human == True)
        & (all_corrs_mm_strong.strong_effect_mouse == True)
        & (all_corrs_mm_strong.geneset_restriction == "both")
    )
]

order = _data_hm.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.pointplot(
    _data_hh,
    order=order,
    x="celltype_combination",
    y="correlation",
    markers=".",
    markersize=3,
    linewidth=0.5,
    color=human_color,
    estimator="median",
    errorbar=("pi", 50),
    ax=ax,
    capsize=0.25,
)
sns.pointplot(
    _data_mm,
    order=order,
    x="celltype_combination",
    y="correlation",
    markers=".",
    markersize=3,
    linewidth=0.5,
    color=mouse_color,
    estimator="median",
    errorbar=("pi", 50),
    ax=ax,
    capsize=0.25,
)
sns.boxplot(
    data = _data_hm,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    boxprops={'edgecolor':'black'},
    whiskerprops = {'color':'black'},
    capprops = {'color':'black'},
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
    # showmeans=True,
    # meanprops={'marker':'_','markerfacecolor':'white','markeredgecolor':'gray','markersize':'4'}
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
# ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
# plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}_strong_in_both_{threshold_expression}.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
_data_hm.loc[_data_hm.num_genes > 10].groupby("cytokine").correlation.min().sort_values(ascending=False)

In [ ]:
_data_hm

In [ ]:
add_barplots = False

threshold_expression = 0.2

combos = [
    ("T_cell_CD4__CD4"),
    ("Treg__Treg"),
    ("Monocyte__CD16_Mono"),
    ("Monocyte__CD14_Mono"),
]

# for celltype_combo in sorted(all_corrs_hm.celltype_combination.unique()):
for celltype_combo in sorted(combos):

    _data = all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.celltype_combination==celltype_combo)
            & (all_corrs_hm_strong.threshold_expression==threshold_expression)
            & (all_corrs_hm_strong.strong_effect_human == True)
            & (all_corrs_hm_strong.strong_effect_mouse == True)
        )
    ]
    corrs_hh = all_corrs_hh_strong.loc[
        (
            (all_corrs_hh_strong.celltype_combination==celltype_combo)
            & (all_corrs_hh_strong.threshold_expression==threshold_expression)
            & (all_corrs_hh_strong.strong_effect_human == True)
            & (all_corrs_hh_strong.strong_effect_mouse == True)
        )
    ]
    corrs_mm = all_corrs_mm_strong.loc[
        (
            (all_corrs_mm_strong.celltype_combination==celltype_combo)
            & (all_corrs_mm_strong.threshold_expression==threshold_expression)
            & (all_corrs_mm_strong.strong_effect_human == True)
            & (all_corrs_mm_strong.strong_effect_mouse == True)
        )
    ]

    
    group_means = _data.num_genes.unique()
    cmap = sns.color_palette("light:#F4D03F", as_cmap=True)
    
    vmin = _data.num_genes.min()
    vmax = _data.num_genes.max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colors = [cmap(norm(v)) for v in group_means]
    order = _data.groupby("cytokine").correlation.median().sort_values(ascending=False).index

    num_humans = count_unique_non_rep_donors(corrs_hh, "human")
    num_humans = add_missing_values(num_humans, order, 1, "cytokine", "num_donors")
    num_humans.loc[:, "species"] = "human"
    num_mice = count_unique_non_rep_donors(corrs_mm, "mouse")
    num_mice = add_missing_values(num_mice, order, 1, "cytokine", "num_donors")
    num_mice.loc[:, "species"] = "mouse"
    num_samples = pd.concat([num_humans.reset_index(), num_mice.reset_index()], axis=0, ignore_index=True)

    if add_barplots:
        fig, ax = plt.subplots(2,1,figsize=(0.105*len(order), 2), height_ratios=[0.2, 0.8], sharex=True)
        ax_top = ax[0]
        ax_bottom = ax[1]
    else:
        fig, ax = plt.subplots(1,1,figsize=(0.105*len(order), 1.3))
        ax_bottom = ax
        
    sns.boxplot(
        data = _data,
        x="cytokine",
        y="correlation",
        hue="num_genes",
        order = order,
        ax=ax_bottom,
        palette=colors,
        # palette="dark:#E7298A",
        # palette=cmap,
        linewidth=0.5,
        linecolor="black",
        legend=None,
        flierprops={
            'marker': 'o',
            'markerfacecolor': 'gray',
            'markeredgecolor': 'black',
            'markersize': 1,
            'linewidth': 0.5,
        }
        
    )
    sns.pointplot(
        corrs_hh,
        order=order,
        x="cytokine",
        y="correlation",
        markers=".",
        markersize=3,
        linewidth=0.5,
        color=human_color,
        estimator="median",
        errorbar=("pi", 50),
        ax=ax_bottom,
    )
    sns.pointplot(
        corrs_mm,
        x="cytokine",
        y="correlation",
        markers=".",
        markersize=3,
        linewidth=0.5,
        color=mouse_color,
        estimator="median",
        errorbar=("pi", 50),
        ax=ax_bottom,
    )
    if add_barplots:
        sns.barplot(
            data=num_samples,
            x="cytokine",
            y="num_donors",
            hue="species",
            order=order,
            ax=ax_top,
            palette={"human": human_color, "mouse": mouse_color},
            legend=False,
        )
    
        ax_top.set_ylim([0, 13])
        ax_top.set_xlabel("")
        ax_top.set_ylabel("# samples", fontsize=6, family="Arial")
        ax_top.set_yticks([0, 3, 6, 9, 12], ["0", "", "6", "", "12"], fontsize=6, family="Arial")
        ax_top.grid(axis="y")
        ax_top.spines[['right', 'top']].set_visible(False)
        ax_top.set_axisbelow(True)
        ax_top.set_title(title_map[celltype_combo], fontsize=6, family="Arial")#, y=1, pad=0)
    else:
        ax_bottom.set_title(title_map[celltype_combo], fontsize=6, family="Arial")#, y=1, pad=0)
    
    # plt.ylim([])
    y_margin = 0.02
    ax_bottom.margins(y=y_margin)
    ax_bottom.set_xticks(np.arange(len(order)), [str(s) for s in order], rotation=90, fontsize=6, family="Arial")
    yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
    ax_bottom.set_yticks(yticks, [str(_) for _ in yticks], fontsize=6, family="Arial")
    ax_bottom.grid(axis="y")
    ygridlines = ax.get_ygridlines()
    # Find the line at the top (highest y value)
    top_y = max([tick.get_ydata()[0] for tick in ygridlines])
    for line in ygridlines:
        # Hide the line at the top
        if line.get_ydata()[0] == top_y:
            line.set_visible(False)
    ax_bottom.set_xlabel("")
    ax_bottom.set_ylabel("correlation", fontsize=6, family="Arial", labelpad=0)
    ax_bottom.spines[['right', 'top']].set_visible(False)
    ax_bottom.margins(y=y_margin)
    ax_bottom.set_ylim([-0.32, 1])

    if False:
        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        cbar = plt.colorbar(sm, ax=ax_bottom, pad=0.01)
        cbar.set_label("num shared genes", rotation=270, labelpad=15, family="Arial", fontsize=6)  # customize label
        cbar.ax.tick_params(labelsize=6)
        for label in cbar.ax.get_yticklabels():
            label.set_fontfamily('Arial')
        # plt.tight_layout()

    filename = f"boxplot_{threshold_expression}_{celltype_combo}_strong_effect_cytokines.svg"
    if add_barplots:
        filename = filename.replace(".svg", "_w_bars.svg")
    plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
    plt.show()

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array(np.array([]))
    
    fig, ax = plt.subplots(1, 1, figsize=(0.1, 1))
    cbar = plt.colorbar(sm, cax=ax)
    
    cbar.set_label("# Genes", rotation=270, labelpad=15, family='Arial', fontsize=6)
    cbar.ax.tick_params(labelsize=6)
    cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
    
    for label in cbar.ax.get_yticklabels():
        label.set_fontfamily('Arial')
    
    plt.tight_layout()
    filename = f"colorbar_num_genes_{threshold_expression}_{celltype_combo}.svg"
    if add_barplots:
        filename = filename.replace(".svg", "_w_bars.svg")
    #plt.savefig(os.path.join(FIGDIR, filename), bbox_inches="tight", pad_inches=0, transparent=True)
    plt.show()

    # break

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[(all_corrs_hm_strong.strong_effect_human==True) & (all_corrs_hm_strong.strong_effect_mouse==True) & (all_corrs_hm_strong.subset_expressed_genes_by=="union")],
    x = "threshold_expression",
    y = "correlation",
    hue="celltype_combination",
    estimator="median",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "T_cell_CD4__CD4")
            & (all_corrs_hm_strong.subset_expressed_genes_by=="union")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="median",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "Monocyte__CD14_Mono")
            & (all_corrs_hm_strong.subset_expressed_genes_by=="union")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="median",
)
plt.show()

### Pseudobulk across all samples (mean per species = True)

##### subset_expressed_genes_by = "union"

In [ ]:
threshold_expression = -1

all_corrs_hm = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == True) 
        & (all_corrs.apply_log2==True)
        & (all_corrs.subset_expressed_genes_by=="union")
        & (all_corrs.donor1.str.startswith("D") & all_corrs.donor2.str.startswith("r"))
    )
]
all_corrs_hm.loc[:, "cytokine"] = all_corrs_hm.loc[:, "cytokine"].apply(replace_greek)

all_corrs_hm_strong = all_corrs_hm.copy()
all_corrs_hm_strong = all_corrs_hm_strong.loc[all_corrs_hm_strong.celltype_combination.apply(get_clean_label_mouse).isin(df_strong_responses_mouse.celltype_mouse.values)]
all_corrs_hm_strong.loc[:, "strong_effect_mouse"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_mouse.set_index("celltype_mouse").loc[get_clean_label_mouse(row.celltype_combination)].cytokines_mouse, 
    axis=1
)

all_corrs_hm_strong.loc[:, "strong_effect_human"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_human.set_index("celltype_human").loc[get_clean_label_human(row.celltype_combination)].cytokines_human, 
    axis=1
)

_data = all_corrs_hm_strong.loc[
    (
        (all_corrs_hm_strong.threshold_expression==threshold_expression)
        & (all_corrs_hm_strong.strong_effect_human == True)
        & (all_corrs_hm_strong.strong_effect_mouse == True)
        & (all_corrs.geneset_restriction=="both")
        & (all_corrs.num_genes >= 10)
    )
].drop_duplicates()
order = _data.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.boxplot(
    data = _data,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}_strong_in_both_mean_per_species_DEGs.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
_data.loc[_data.cytokine == "IL-10"]

In [ ]:
_data.groupby("cytokine").correlation.min().sort_values()

In [ ]:
__data = _data.loc[_data.num_genes >= 5].drop_duplicates()

df_pivot = __data.pivot(
    index="cytokine",
    columns="celltype_combination",
    values="correlation",
)

def sort_key(label):
    if label.startswith('IL-'):
        parts = label.split('-')
        if len(parts) == 2 and parts[1].isdigit():
            return (parts[0], int(parts[1]))
    return (label, 0)

df_pivot = df_pivot.sort_index(key=lambda index: [sort_key(x) for x in index])
order_x = df_pivot.columns[df_pivot.columns.str.lower().argsort()]

order_y = __data.groupby("cytokine").correlation.median().sort_values(ascending=False).index

df_pivot = df_pivot.loc[order_y, order_x]

def fmt_no_leading_zero(x, digits=2):
    s = f"{x:.{digits}f}"
    if s in ["0.0", "-0.0"]:
        return "0"
    if s.startswith("-0"):
        return s.replace("-0", "-")
    elif s.startswith("0"):
        return s[1:]
    return s

df_annot = df_pivot.copy()
for row in range(df_annot.shape[0]):
    for col in range(df_annot.shape[1]):
        v = df_annot.iloc[row, col]
        if np.isnan(v):
            df_annot.iloc[row, col] = ""
        else:
            df_annot.iloc[row, col] = fmt_no_leading_zero(v, 1)

fig, axs = plt.subplots(1,2,figsize=(1.9, 2.7), width_ratios=(0.8, 0.2))
ax = axs[0]
sns.heatmap(
    data=df_pivot,
    cmap="RdBu_r",
    vmin=-1,
    center=0,
    vmax=1,
    square=False,
    ax=ax,
    cbar=False,
    linewidth=0.5,
    linecolor="white",
    annot=df_annot,
    annot_kws={"fontsize": 5, "family": "Arial"},
    fmt="",
)

def format_celltype_labels(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_celltype_labels(l) for l in x]
    if not isinstance(x, str):
        x = x.get_text()
    x = x.replace("B", "B cell")
    x = x.replace("CD56-bright_NK", "NK CD56hi")
    x = x.replace("CD56-dim_NK", "NK CD56low")
    x = x.replace("CD4", "CD4 T cell")
    x = x.replace("CD8", "CD8 T cell")
    x = x.replace("CD14_Mono", "CD14 Mono")
    x = x.replace("CD16_Mono", "CD16 Mono")
    return x

ax.set_facecolor("lightgray")
# ax.set_xticks(0.5+np.arange(df_pivot.shape[1]), [format_celltype_labels(l.get_text().split("__")[1]) for l in ax.get_xticklabels()], fontsize=6, family="Arial")
ax.set_xticks(0.5+np.arange(df_pivot.shape[1]), [format_celltype_labels(l.split("__")[1]) for l in df_pivot.columns], fontsize=6, family="Arial")
ax.set_yticks(0.5+np.arange(df_pivot.shape[0]), [replace_greek(l) for l in df_pivot.index], fontsize=6, family="Arial")
ax.tick_params(length=0)
ax.set_xlabel("")
ax.set_ylabel("")


ax_bar = axs[1]
sns.boxplot(
    data=__data,
    y="cytokine",
    x="correlation",
    ax=ax_bar,
    order = df_pivot.index,
    color="gold",
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
ax_bar.set_ylabel("")
ax_bar.set_xlabel(r"$R$", fontsize=6, family="Arial")
ax_bar.set_yticks([])
ax_bar.grid(axis="x", color="lightgray", linewidth=0.5)
ax_bar.spines[['right', 'top']].set_visible(False)
xticks = [0, 0.25, 0.5, 0.75]
xticklabels = ["0", "", "0.5", ""]
ax_bar.set_xticks(xticks, xticklabels, family="Arial", fontsize=6, rotation=0, ha="center")
# y_margin = 0.02
# ax_bar.margins(y=y_margin)

box = ax_bar.get_position()
box.x0 = box.x0 - 0.06
box.x1 = box.x1 - 0.06
ax_bar.set_position(box)

plt.savefig(os.path.join(FIGDIR, f"heatmap_correlations_strong_responses_DEGs.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
df_pivot = _data.loc[_data.num_genes >= 5].drop_duplicates().pivot(
    index="cytokine",
    columns="celltype_combination",
    values="correlation",
)

def sort_key(label):
    if label.startswith('IL-'):
        parts = label.split('-')
        if len(parts) == 2 and parts[1].isdigit():
            return (parts[0], int(parts[1]))
    return (label, 0)

df_pivot = df_pivot.sort_index(key=lambda index: [sort_key(x) for x in index])
order_x = df_pivot.columns[df_pivot.columns.str.lower().argsort()]

df_pivot = df_pivot.loc[:, order_x]

def fmt_no_leading_zero(x, digits=2):
    s = f"{x:.{digits}f}"
    if s in ["0.0", "-0.0"]:
        return "0"
    if s.startswith("-0"):
        return s.replace("-0", "-")
    elif s.startswith("0"):
        return s[1:]
    return s

df_annot = df_pivot.copy()
for row in range(df_annot.shape[0]):
    for col in range(df_annot.shape[1]):
        v = df_annot.iloc[row, col]
        if np.isnan(v):
            df_annot.iloc[row, col] = ""
        else:
            df_annot.iloc[row, col] = fmt_no_leading_zero(v, 1)

fig, ax = plt.subplots(1,1,figsize=(3, 2.7))
sns.heatmap(
    data=df_pivot,
    cmap="RdBu_r",
    vmin=-1,
    center=0,
    vmax=1,
    square=True,
    ax=ax,
    cbar=False,
    linewidth=0.5,
    linecolor="white",
    annot=df_annot,
    annot_kws={"fontsize": 5, "family": "Arial"},
    fmt="",
)

def format_celltype_labels(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_celltype_labels(l) for l in x]
    if not isinstance(x, str):
        x = x.get_text()
    x = x.replace("B", "B cell")
    x = x.replace("CD56-bright_NK", "NK CD56hi")
    x = x.replace("CD56-dim_NK", "NK CD56low")
    x = x.replace("CD4", "CD4 T cell")
    x = x.replace("CD8", "CD8 T cell")
    x = x.replace("CD14_Mono", "CD14 Mono")
    x = x.replace("CD16_Mono", "CD16 Mono")
    return x

ax.set_facecolor("lightgray")
ax.set_xticks(0.5+np.arange(df_pivot.shape[1]), [format_celltype_labels(l.get_text().split("__")[1]) for l in ax.get_xticklabels()], fontsize=6, family="Arial")
ax.set_yticks(0.5+np.arange(df_pivot.shape[0]), [replace_greek(l) for l in df_pivot.index], fontsize=6, family="Arial")
ax.tick_params(length=0)
plt.xlabel("")
plt.ylabel("")
plt.savefig(os.path.join(FIGDIR, f"heatmap_correlations_strong_responses_DEGs.svg"), pad_inches=0, bbox_inches="tight", transparent=False)
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch

df = df_pivot.fillna(0)

# Compute row and column linkage
row_linkage = sch.linkage(df, method="average", metric="correlation")
col_linkage = sch.linkage(df.T, method="average", metric="correlation")

# Get reordered indices
row_order = sch.leaves_list(row_linkage)
col_order = sch.leaves_list(col_linkage)

# Reorder DataFrame
df_clustered = df.iloc[row_order, col_order]

# Plot reordered heatmap without dendrograms
sns.heatmap(df_clustered, cmap="RdBu_r", center=0)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "T_cell_CD4__CD4")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="mean",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "Monocyte__CD14_Mono")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="mean",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="celltype_combination",
    estimator="mean",
)
plt.show()

#### subset_expressed_genes_by = "intersection"

In [ ]:
threshold_expression = 0.5

all_corrs_hm = all_corrs.loc[
    (
        (all_corrs.comparison == "human_vs_mouse") 
        & (all_corrs.method == "pearson") 
        & (all_corrs.mean_per_species == True) 
        & (all_corrs.apply_log2==True)
        & (all_corrs.subset_expressed_genes_by=="intersection")
        & (all_corrs.donor1.str.startswith("D") & all_corrs.donor2.str.startswith("r"))
    )
]
all_corrs_hm.loc[:, "cytokine"] = all_corrs_hm.loc[:, "cytokine"].apply(replace_greek)

all_corrs_hm_strong = all_corrs_hm.copy()
all_corrs_hm_strong = all_corrs_hm_strong.loc[all_corrs_hm_strong.celltype_combination.apply(get_clean_label_mouse).isin(df_strong_responses_mouse.celltype_mouse.values)]
all_corrs_hm_strong.loc[:, "strong_effect_mouse"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_mouse.set_index("celltype_mouse").loc[get_clean_label_mouse(row.celltype_combination)].cytokines_mouse, 
    axis=1
)

all_corrs_hm_strong.loc[:, "strong_effect_human"] = all_corrs_hm_strong.apply(
    lambda row: row.cytokine in df_strong_responses_human.set_index("celltype_human").loc[get_clean_label_human(row.celltype_combination)].cytokines_human, 
    axis=1
)

_data = all_corrs_hm_strong.loc[
    (
        (all_corrs_hm_strong.threshold_expression==threshold_expression)
        & (all_corrs_hm_strong.strong_effect_human == True)
        & (all_corrs_hm_strong.strong_effect_mouse == True)
    )
]
order = _data.groupby("celltype_combination").correlation.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.3))
sns.boxplot(
    data = _data,
    x="celltype_combination",
    y="correlation",
    hue="celltype_combination",
    palette=celltype_color_mapping,
    order=order,
    ax=ax,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
)
yticks = [-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")
ax.set_xticks(np.arange(len(order)), [title_map_short[s] for s in order], rotation=90, fontsize=6, family="Arial")
ax.set_ylim([-0.32, 1.])
ax.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
ax.set_xlabel("")
ax.set_ylabel("", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
y_margin = 0.02
ax.margins(y=y_margin)
plt.title("Median correlations\nper cell type", fontsize=6, family="Arial")
plt.savefig(os.path.join(FIGDIR, f"boxplot_per_celltype_{threshold_expression}_strong_in_both_mean_per_species.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()   

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "T_cell_CD4__CD4")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="median",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
            & (all_corrs_hm_strong.celltype_combination == "Monocyte__CD14_Mono")
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="cytokine",
    estimator="median",
)
plt.show()

In [ ]:
sns.pointplot(
    data=all_corrs_hm_strong.loc[
        (
            (all_corrs_hm_strong.strong_effect_human==True) 
            & (all_corrs_hm_strong.strong_effect_mouse==True) 
        )
    ],
    x = "threshold_expression",
    y = "correlation",
    hue="celltype_combination",
    estimator="median",
)
plt.show()

In [ ]:
df_response_magnitudes_human = pd.read_excel("/Users/soeren.becker/repos/HumanCytokineDict/data/SI_tables.xlsx", sheet_name="2.response_magnitudes")
df_response_magnitudes_human = pd.melt(df_response_magnitudes_human, id_vars=['Cytokine'], var_name='Celltype', value_name='Response_magnitude_human')
df_response_magnitudes_human.loc[df_response_magnitudes_human.Celltype == 'NK CD56hi', "Celltype"] = "NK cell"
df_response_magnitudes_human.loc[:, "species"] = 'human'
df_response_magnitudes_human.Celltype.unique()

In [ ]:
df_response_magnitudes_mouse = load_response_magnitudes_mouse()
df_response_magnitudes_mouse.loc[:, "Response_magnitude_mouse"] = df_response_magnitudes_mouse.loc[:, "Magnitude_norm_float"] / 100
df_response_magnitudes_mouse.loc[df_response_magnitudes_mouse.Celltype == 'CD4+ T cell', "Celltype"] = 'CD4 T cell'
df_response_magnitudes_mouse.loc[df_response_magnitudes_mouse.Celltype == 'CD8+ T cell', "Celltype"] = 'CD8 T cell'
df_response_magnitudes_mouse.loc[df_response_magnitudes_mouse.Celltype == 'cDC1', "Celltype"] = 'cDC'
df_response_magnitudes_mouse.loc[:, "species"] = 'mouse'
df_response_magnitudes_mouse.Celltype.unique()

In [ ]:
shared_cytokines = np.intersect1d(df_response_magnitudes_mouse.Cytokine.unique(), df_response_magnitudes_human.Cytokine.unique())
shared_cytokines

In [ ]:
shared_celltypes = np.intersect1d(df_response_magnitudes_mouse.Celltype.unique(), df_response_magnitudes_human.Celltype.unique())
shared_celltypes

In [ ]:
df_response_magnitudes_human.loc[
    (
        df_response_magnitudes_human.Celltype.isin(shared_celltypes)
        & df_response_magnitudes_human.Cytokine.isin(shared_cytokines)
    ),
    ["species", "Celltype", "Cytokine", "Response_magnitude_human"]
].Celltype.unique()

In [ ]:
responses_both = pd.merge(
    left=df_response_magnitudes_mouse.loc[
            (
                df_response_magnitudes_mouse.Celltype.isin(shared_celltypes) 
                & df_response_magnitudes_mouse.Cytokine.isin(shared_cytokines)
            ),
            ["species", "Celltype", "Cytokine", "Response_magnitude_mouse"]
        ], 
    right=df_response_magnitudes_human.loc[
            (
                df_response_magnitudes_human.Celltype.isin(shared_celltypes)
                & df_response_magnitudes_human.Cytokine.isin(shared_cytokines)
            ),
            ["species", "Celltype", "Cytokine", "Response_magnitude_human"]
        ],
    on=["Celltype", "Cytokine"],
    how="inner"
    
    #axis=1
)

In [ ]:
df_response_magnitudes_mouse.loc[
    (
        df_response_magnitudes_mouse.Celltype.isin(shared_celltypes) 
        & df_response_magnitudes_mouse.Cytokine.isin(shared_cytokines)
    ),
    ["species", "Celltype", "Cytokine", "Response_magnitude_mouse"]
].Celltype.unique()

In [ ]:
sns.scatterplot(
    responses_both.loc[responses_both.Celltype != "pDC"],
    x="Response_magnitude_mouse",
    y="Response_magnitude_human",
    hue="Celltype"
)
plt.show()

In [ ]:
responses_both

In [ ]:
def format_celltype_labels(x):
    if isinstance(x, (list, np.ndarray, pd.Index, pd.Series)):
        return [format_celltype_labels(l) for l in x]
    if not isinstance(x, str):
        x = x.get_text()
    x = x.replace("B", "B cell")
    x = x.replace("CD56-bright_NK", "NK CD56hi")
    x = x.replace("CD56-dim_NK", "NK CD56low")
    x = x.replace("CD4", "CD4 T cell")
    x = x.replace("CD8", "CD8 T cell")
    x = x.replace("CD14_Mono", "CD14 Mono")
    x = x.replace("CD16_Mono", "CD16 Mono")
    return x

In [ ]:
_data.loc[:, "formated_celltype"] = format_celltype_labels(_data.celltype_combination.str.split("__").str.get(1).values)

In [ ]:
for row_i, row in responses_both.loc[responses_both.Celltype != "pDC"].iterrows():
    _corrs = _data.loc[(_data.drop_duplicates().formated_celltype == row.Celltype) & (_data.cytokine == row.Cytokine)].correlation.values
    if len(_corrs) > 0:
        responses_both.loc[row_i, "correlation"] = _corrs.mean()
    

In [ ]:
responses_both

In [ ]:
sns.scatterplot(
    responses_both.loc[responses_both.Celltype != "pDC"],
    x="Response_magnitude_mouse",
    y="Response_magnitude_human",
    hue="correlation"
)
plt.show()

In [ ]:
spearmanr(
    (responses_both.loc[~responses_both.correlation.isna()].Response_magnitude_mouse * responses_both.loc[~responses_both.correlation.isna()].Response_magnitude_human).values,
    responses_both.loc[~responses_both.correlation.isna()].correlation.values,
)[0]

In [ ]:
hcd_sig.cytokine.unique()

In [ ]:
hcd_sig.loc[
    (
        (hcd_sig.celltype_old == celltype_combination[1]) 
        & (hcd_sig.cytokine == cytokine)
        & (hcd_sig.log_fc.abs() >= 1)
    )
].gene.values,

In [ ]:
_adata_mouse = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/mouse_cytokine_dict/mouse_cytokine_dict_all.h5ad")
_adata_mouse = map_gene_names(_adata_mouse, mouse_to_human_dict)

In [ ]:
all_homolog_mouse_genes = _adata_mouse.var_names
len(all_homolog_mouse_genes)

In [ ]:
celltype_combination = celltype_combinations[0]
threshold_high = 1
threshold_low = 0.25

df_large_fcs = pd.DataFrame()

hcd_sig.loc[:, "cytokine_nogreek"] = hcd_sig.loc[:, "cytokine"].apply(replace_greek)

celltype_combo_to_names_map = {
    'B cell__B cell': ("B_cell", "B"),
    'CD4 T cell__CD4 T cell': ('T_cell_CD4', 'CD4'),
    'CD8 T cell__CD8 T cell': ('T_cell_CD8', 'CD8'),
    'CD14 Mono__Monocyte': ('Monocyte', 'CD14_Mono'),
    'CD16 Mono__Monocyte': ('Monocyte', 'CD16_Mono'),
    'NK CD56hi__NK cell': ('NK_cell', 'CD56-bright_NK'),
    'NK CD56low__NK cell': ('NK_cell', 'CD56-dim_NK'),
    'Treg__Treg': ('Treg', 'Treg'),
    'cDC__cDC1': ('cDC1', 'cDC'),
    'cDC__cDC2': ('cDC2', 'cDC'),
}

for celltype_combo in tqdm(df_strong_responses.loc[df_strong_responses.species == "overlap"].celltype_combo.unique()):
    strong_cytokines = df_strong_responses.loc[(df_strong_responses.species == "overlap") & (df_strong_responses.celltype_combo == celltype_combo)].overlap_cytokines.values
    if len(strong_cytokines) == 0:
        print(celltype_combo,strong_cytokines) 
        continue
    strong_cytokines = strong_cytokines[0]
    for cytokine in tqdm(strong_cytokines, leave=False):

        celltype_mouse, celltype_human = celltype_combo_to_names_map[celltype_combo]
        
        mcd = get_mouse_dict(
            cytokine_name_map=cytokine_name_map,
            celltype=celltype_mouse,
            cytokine=cytokine,
            mouse_to_human_dict=mouse_to_human_dict, 
            subset_to_homologs=False,
        )
        
        large_fcs_human = hcd_sig.loc[
            (
                (hcd_sig.celltype_old == celltype_human) 
                & (hcd_sig.cytokine_nogreek == cytokine)
                & (hcd_sig.log_fc.abs() >= threshold_high)
            )
        ]

        large_fcs_human_homologs = large_fcs_human.loc[large_fcs_human.gene.isin(all_homolog_mouse_genes)]
        
        above_low_fcs_mouse = mcd.loc[(mcd.log_fc.abs()>=threshold_low) & (~mcd.gene.isna())]
        
        row_idx = df_large_fcs.shape[0]
        df_large_fcs.loc[row_idx, "threshold_high"] = threshold_high
        df_large_fcs.loc[row_idx, "threshold_low"] = threshold_low
        df_large_fcs.loc[row_idx, "cytokine"] = cytokine
        df_large_fcs.loc[row_idx, "celltype_human"] = celltype_human
        df_large_fcs.loc[row_idx, "celltype_mouse"] = celltype_mouse
        
        df_large_fcs.loc[row_idx, "num_large_lfc_human_all"] = len(large_fcs_human) 
        df_large_fcs.loc[row_idx, "num_large_lfc_human_homolog_mappable"] = len(large_fcs_human_homologs) 
        df_large_fcs.loc[row_idx, "num_large_lfc_missing_all_human"] = len(large_fcs_human) - len(np.intersect1d(large_fcs_human.gene.values, above_low_fcs_mouse.gene.values))
        df_large_fcs.loc[row_idx, "num_large_lfc_missing_homolog_mappable_human"] = len(large_fcs_human_homologs) - len(np.intersect1d(large_fcs_human_homologs.gene.values, above_low_fcs_mouse.gene.values))
        df_large_fcs.loc[row_idx, "num_above_low_lfc_mouse"] = len(above_low_fcs_mouse)

        large_fcs_mouse_homologs = mcd.loc[mcd.gene.isin(all_homolog_mouse_genes) & (mcd.log_fc.abs()>=threshold_high)].gene.values
        above_low_fcs_human = hcd.loc[(hcd.log_fc.abs()>=threshold_low) & (~hcd.gene.isna())].gene.values
        df_large_fcs.loc[row_idx, "num_large_lfc_missing_homolog_mappable_mouse"] = len(large_fcs_mouse_homologs) - len(np.intersect1d(large_fcs_mouse_homologs, above_low_fcs_human))
        

In [ ]:
# df_large_fcs.to_csv("./df_large_fcs.csv")

In [ ]:
df_large_fcs.groupby(["celltype_human", "celltype_mouse"]).num_large_lfc_missing_homolog_mappable_human.median().median()

In [ ]:
_df_large_fcs = df_large_fcs.copy()
_df_large_fcs.loc[_df_large_fcs.celltype_mouse == "cDC1", "celltype_human"] = "cDC1"
_df_large_fcs.loc[_df_large_fcs.celltype_mouse == "cDC2", "celltype_human"] = "cDC2"

order = _df_large_fcs.groupby("celltype_human").num_large_lfc_missing_homolog_mappable_human.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.1))
sns.boxplot(
    data=_df_large_fcs,
    x="celltype_human",
    y="num_large_lfc_missing_homolog_mappable_human",
    hue="celltype_human",
    order=order,
    palette=celltype_color_mapping,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
    ax=ax,
)

yticks = [0, 200, 400, 600, 800]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")

def format_celltype_label(x):
    celltype_combo_to_names_map = {
        "B": "B_cell",
        'CD4': 'CD4 T cell',
        'CD8': 'CD8 T cell',
        'CD14_Mono': 'CD14 Mono',
        'CD16_Mono': 'CD16 Mono',
        'CD56-bright_NK': "NK CD56hi",
        'CD56-dim_NK': "NK CD56low",
    }
    return celltype_combo_to_names_map.get(x,x)

ax.set_yscale("log")
ax.set_xticks(np.arange(len(order)), [format_celltype_label(s) for s in order], rotation=90, fontsize=6, family="Arial")
ax.set_ylim([0, 950])
ax.grid(axis="y")
ax.set_xlabel("")
# ax.set_ylabel("# discordant genes \n|lfc(human)| >> |lfc(mouse)|", fontsize=6, family="Arial")
ax.set_ylabel("# discordant genes", fontsize=6, family="Arial")
ax.set_title("|lfc(human)| >> |lfc(mouse)|", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
ax.grid(axis="y")
y_margin = 0.02
ax.margins(y=y_margin)

plt.grid(axis="y")
plt.savefig(os.path.join(FIGDIR, f"boxplot_missing_genes_human_in_mouse.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
_df_large_fcs = df_large_fcs.copy()
_df_large_fcs.loc[_df_large_fcs.celltype_mouse == "cDC1", "celltype_human"] = "cDC1"
_df_large_fcs.loc[_df_large_fcs.celltype_mouse == "cDC2", "celltype_human"] = "cDC2"

#order = _df_large_fcs.groupby("celltype_human").num_large_lfc_missing_homolog_mappable_mouse.median().sort_values(ascending=False).index
fig, ax = plt.subplots(1,1,figsize=(0.1*(len(order)), 1.1))
sns.boxplot(
    data=_df_large_fcs,
    x="celltype_human",
    y="num_large_lfc_missing_homolog_mappable_mouse",
    hue="celltype_human",
    order=order,
    palette=celltype_color_mapping,
    flierprops={
        'marker': 'o',
        'markerfacecolor': 'gray',
        'markeredgecolor': 'black',
        'markersize': 1,
        'linewidth': 0.5,
    },
    linewidth=0.5,
    ax=ax,
)

yticks = [0, 2, 4, 6, 8, 10]
ax.set_yticks(yticks, [str(t) for t in yticks], fontsize=6, family="Arial")

def format_celltype_label(x):
    celltype_combo_to_names_map = {
        "B": "B_cell",
        'CD4': 'CD4 T cell',
        'CD8': 'CD8 T cell',
        'CD14_Mono': 'CD14 Mono',
        'CD16_Mono': 'CD16 Mono',
        'CD56-bright_NK': "NK CD56hi",
        'CD56-dim_NK': "NK CD56low",
    }
    return celltype_combo_to_names_map.get(x,x)

#ax.set_yscale("log")
ax.set_xticks(np.arange(len(order)), [format_celltype_label(s) for s in order], rotation=90, fontsize=6, family="Arial")
#ax.set_ylim([0, 950])
ax.grid(axis="y")
ax.set_xlabel("")
# ax.set_ylabel("# discordant genes \n|lfc(mouse)| >> |lfc(human)|", fontsize=6, family="Arial")
ax.set_ylabel("# discordant genes", fontsize=6, family="Arial")
ax.set_title("|lfc(mouse)| >> |lfc(human)|", fontsize=6, family="Arial")
ax.spines[['right', 'top']].set_visible(False)
ax.grid(axis="y")
plt.grid(axis="y")
ygridlines = ax.get_ygridlines()
# Find the line at the top (highest y value)
top_y = max([tick.get_ydata()[0] for tick in ygridlines])
for line in ygridlines:
    # Hide the line at the top
    if line.get_ydata()[0] == top_y:
        line.set_visible(False)
y_margin = 0.02
#ax.margins(y=y_margin)


plt.savefig(os.path.join(FIGDIR, f"boxplot_missing_genes_mouse_in_human.svg"), pad_inches=0, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
df_large_fcs

In [ ]:
df_large_fcs.groupby(["celltype_human", "celltype_mouse"]).num_large_lfc_missing_homolog_mappable_mouse.max()

In [ ]:
df_large_fcs.pivot(
    index="cytokine",
    columns=["celltype_human", "celltype_mouse"],
    values="num_missing"
)

In [ ]:
df_strong_responses

### with re-computed response magnitudes

In [ ]:
response_magnitures_mouse = pd.read_csv("./response_magnitudes_mouse.csv", index_col=0)
values = response_magnitures_mouse.loc[response_magnitures_mouse.index != "PBS", "T_cell_CD4"]
q = values.quantile(0.35)
threshold = np.mean(values < q)
plt.hist(values, bins=20)
plt.plot([])
plt.show()

plt.plot(sorted(response_magnitures_mouse.loc[:, "T_cell_CD4"]))

plt.plot(sorted(response_magnitures_mouse.loc[:, "T_cell_CD8"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "Monocyte"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "Treg"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "NK_cell"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "B_cell"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "pDC"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "cDC1"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "cDC2"]))
plt.plot(sorted(response_magnitures_mouse.loc[:, "ILC"]))
plt.show()

strong_magnitude_cytokines_per_celltype = {}

for celltype in response_magnitures_mouse.columns:
    strong_magnitude_cytokines_per_celltype[celltype] = response_magnitures_mouse.loc[response_magnitures_mouse.loc[:, celltype] > 0.7].index.tolist()

all_corrs_mm.loc[:, "strong_response"] = all_corrs_mm.apply(lambda row: row.cytokine in strong_magnitude_cytokines_per_celltype[row.celltype_combination.split("__")[0]], axis=1)
all_corrs_hh.loc[:, "strong_response"] = all_corrs_hh.apply(lambda row: row.cytokine in strong_magnitude_cytokines_per_celltype[row.celltype_combination.split("__")[0]], axis=1)
all_corrs_hm.loc[:, "strong_response"] = all_corrs_hm.apply(lambda row: row.cytokine in strong_magnitude_cytokines_per_celltype[row.celltype_combination.split("__")[0]], axis=1)

fig, ax = plt.subplots(1,1,figsize=(6, 3))
sns.boxplot(
    data=all_corrs_hm.loc[(all_corrs_hm.threshold_expression==threshold_expression) & (all_corrs_hm.strong_response == True)],
    x="celltype_combination",
    y="correlation",
    ax=ax
)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), fontsize=8, family="Arial", rotation=60)
plt.xlabel("")
plt.title("human vs mouse")
plt.show()

fig, ax = plt.subplots(1,1,figsize=(6, 3))
sns.boxplot(
    data=all_corrs_mm.loc[(all_corrs_mm.threshold_expression==threshold_expression) & (all_corrs_mm.strong_response == True)],
    x="celltype_combination",
    y="correlation",
    ax=ax
)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), fontsize=8, family="Arial", rotation=60)
plt.xlabel("")
plt.title("mouse vs mouse")
plt.show()

fig, ax = plt.subplots(1,1,figsize=(6, 3))
sns.boxplot(
    data=all_corrs_hh.loc[(all_corrs_hh.threshold_expression==threshold_expression) & (all_corrs_hh.strong_response == True)],
    x="celltype_combination",
    y="correlation",
    ax=ax
)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), fontsize=8, family="Arial", rotation=60)
plt.xlabel("")
plt.title("human vs human")
plt.show()